# DG16M Grasp Dataset Visualizer

This notebook loads the generated dual-arm grasp datasets and plots the object mesh alongside the gripper poses and contact points using matplotlib.

In [120]:
import h5py
import numpy as np
import trimesh

In [121]:
mesh_path = "../data/1a30adabf5a2bb848af30108ea9ccb6c.obj"
grasp_path = "output/grasps/1a30adabf5a2bb848af30108ea9ccb6c.h5"
gripper_path = "/home/sidd-zeppelin/Public/research/dg16m/DG16M-dataset/gripper.obj"
max_grasps = 25

In [122]:
mesh = trimesh.load(mesh_path)
if isinstance(mesh, trimesh.Scene):
    mesh = mesh.dump(concatenate=True)
mesh.apply_translation(-mesh.centroid)

with h5py.File(grasp_path, 'r') as grasp_file:
    scale = grasp_file['object/scale'][()]
    mesh.apply_scale(scale)
    grasps = grasp_file['grasps/grasps'][:]
    passing_indices = grasp_file['grasps/fc_passing_indices'][:]
    failed_indices = grasp_file['grasps/fc_failed_indices'][:]
    contact_points = grasp_file['grasps/contact_points'][:]

if max_grasps is not None:
    if len(passing_indices) > max_grasps:
        passing_indices = np.random.choice(passing_indices, max_grasps, replace=False)
    if len(failed_indices) > max_grasps:
        failed_indices = np.random.choice(failed_indices, max_grasps, replace=False)

gripper = trimesh.load(gripper_path)
if isinstance(gripper, trimesh.Scene):
    gripper = gripper.dump(concatenate=True)

gripper_vertices = gripper.vertices
gripper_edges = gripper.edges_unique

In [ ]:
mesh = trimesh.load(mesh_path)
if isinstance(mesh, trimesh.Scene):
    mesh = mesh.dump(concatenate=True)

with h5py.File(grasp_path, 'r') as grasp_file:
    scale = grasp_file['object/scale'][()]
    mesh.apply_scale(scale)
    mesh.apply_translation(-mesh.centroid)
    
    grasps = grasp_file['grasps/grasps'][:]
    passing_indices = grasp_file['grasps/fc_passing_indices'][:]
    failed_indices = grasp_file['grasps/fc_failed_indices'][:]
    contact_points = grasp_file['grasps/contact_points'][:]

gripper = trimesh.load(gripper_path)
if isinstance(gripper, trimesh.Scene):
    gripper = gripper.dump(concatenate=True)

def visualize_grasps(num_passing=0, num_failing=0):
    geometries = [mesh]
    
    R_rot = trimesh.transformations.rotation_matrix(np.pi / 2, [1, 0, 0])
    T_trans = trimesh.transformations.translation_matrix([0, 0, 0.04467])
    R = T_trans @ R_rot

    pass_idx = np.random.choice(passing_indices, min(num_passing, len(passing_indices)), replace=False)
    fail_idx = np.random.choice(failed_indices, min(num_failing, len(failed_indices)), replace=False)

    for idx in pass_idx:
        g_left = gripper.copy()
        g_left.visual = trimesh.visual.ColorVisuals(g_left)
        g_left.visual.face_colors = [0, 255, 0, 30]
        g_left.apply_transform(R)
        g_left.apply_transform(grasps[idx, 0])
        
        g_right = gripper.copy()
        g_right.visual = trimesh.visual.ColorVisuals(g_right)
        g_right.visual.face_colors = [0, 255, 0, 30]
        g_right.apply_transform(R)
        g_right.apply_transform(grasps[idx, 1])
        
        geometries.extend([g_left, g_right])
        
        for cp in contact_points[idx]:
            sphere = trimesh.creation.uv_sphere(radius=0.003)
            sphere.visual.face_colors = [0, 255, 0, 255]
            sphere.apply_translation(cp)
            geometries.append(sphere)

    for idx in fail_idx:
        g_left = gripper.copy()
        g_left.visual = trimesh.visual.ColorVisuals(g_left)
        g_left.visual.face_colors = [255, 0, 0, 15]
        g_left.apply_transform(R)
        g_left.apply_transform(grasps[idx, 0])
        
        g_right = gripper.copy()
        g_right.visual = trimesh.visual.ColorVisuals(g_right)
        g_right.visual.face_colors = [255, 0, 0, 15]
        g_right.apply_transform(R)
        g_right.apply_transform(grasps[idx, 1])
        
        geometries.extend([g_left, g_right])
        
        for cp in contact_points[idx]:
            sphere = trimesh.creation.uv_sphere(radius=0.002)
            sphere.visual.face_colors = [255, 0, 0, 255]
            sphere.apply_translation(cp)
            geometries.append(sphere)

    scene = trimesh.Scene(geometries)
    return scene.show()


## Visualize One Passing Grasp


In [ ]:
visualize_grasps(num_passing=5, num_failing=0)


: 

## Visualize One Failing Grasp


In [125]:
visualize_grasps(num_passing=0, num_failing=1)


## Visualize All Grasps (25 passing, 25 failing)


In [126]:
visualize_grasps(num_passing=25, num_failing=25)
